# Torus Subject-Wise Split Creation

This notebook creates train/test/val OBJ split JSON files for the torus longitudinal dataset
using only `labels.pt` and OBJ filenames in `/home/jakaria/torus_creation/torus_mesh/obj_files`.

Splits are subject-wise: all timepoints of a subject stay in one split.

Output directory:
`../examples/splits/splits_torus`


In [2]:
from pathlib import Path
import json
import random
import re
import torch

labels_pt = Path('/home/jakaria/torus_creation/torus_mesh/obj_files/labels.pt')
obj_dir = labels_pt.parent
output_dir = Path('../examples/splits/splits_torus')

train_ratio = 0.80
test_ratio = 0.15
val_ratio = 0.05
random_seed = 42

assert abs(train_ratio + test_ratio + val_ratio - 1.0) < 1e-6
assert labels_pt.exists(), f'Missing labels file: {labels_pt}'
assert obj_dir.exists(), f'Missing OBJ directory: {obj_dir}'
output_dir.mkdir(parents=True, exist_ok=True)


In [3]:
labels = torch.load(labels_pt, map_location='cpu')
assert isinstance(labels, dict), f'labels.pt must be dict, got {type(labels)}'

all_obj_files = sorted(p.name for p in obj_dir.glob('*.obj'))
all_obj_set = set(all_obj_files)

base_pattern = re.compile(r'^ID_(\d+)_t(\d+)$')

label_bases = []
for k in labels.keys():
    if not isinstance(k, str):
        continue
    if base_pattern.match(k):
        label_bases.append(k)

label_bases = sorted(set(label_bases))
assert len(label_bases) > 0, 'No torus keys like ID_###_t# found in labels.pt'

label_obj_files = sorted(f'{b}.obj' for b in label_bases)
label_obj_set = set(label_obj_files)

missing_obj_from_labels = sorted(label_obj_set - all_obj_set)
extra_obj_without_labels = sorted(all_obj_set - label_obj_set)

assert not missing_obj_from_labels, (
    'OBJ files missing for some labels, e.g.: ' + ', '.join(missing_obj_from_labels[:5])
)

if extra_obj_without_labels:
    print(f'Warning: {len(extra_obj_without_labels)} OBJ files have no label key (ignored).')

subject_to_files = {}
subject_to_tp = {}

for base in label_bases:
    m = base_pattern.match(base)
    sid = str(int(m.group(1)))
    tp = int(m.group(2))
    fname = f'{base}.obj'
    subject_to_files.setdefault(sid, []).append(fname)
    subject_to_tp.setdefault(sid, []).append(tp)

for sid in subject_to_files:
    subject_to_files[sid] = sorted(
        subject_to_files[sid],
        key=lambda x: int(base_pattern.match(Path(x).stem).group(2)),
    )

subjects = sorted(subject_to_files.keys(), key=int)

print(f'labels.pt keys used: {len(label_bases)}')
print(f'OBJ files in directory: {len(all_obj_files)}')
print(f'Subjects found: {len(subjects)}')


labels.pt keys used: 500
OBJ files in directory: 500
Subjects found: 100


In [4]:
tp_counts = sorted(len(v) for v in subject_to_files.values())
min_tp = tp_counts[0]
max_tp = tp_counts[-1]

print(f'Timepoints per subject: min={min_tp}, max={max_tp}')
if min_tp != max_tp:
    print('Warning: not all subjects have the same number of timepoints.')

# Optional strict check for this dataset (100 subjects x 5 timepoints).
if len(subjects) == 100:
    print('Subject count is 100 (expected).')
if min_tp == 5 and max_tp == 5:
    print('Each subject has 5 timepoints (expected).')


Timepoints per subject: min=5, max=5
Subject count is 100 (expected).
Each subject has 5 timepoints (expected).


In [5]:
rng = random.Random(random_seed)
rng.shuffle(subjects)

num_subjects = len(subjects)
num_train = int(num_subjects * train_ratio)
num_test = int(num_subjects * test_ratio)
num_val = num_subjects - num_train - num_test

train_subjects = subjects[:num_train]
test_subjects = subjects[num_train:num_train + num_test]
val_subjects = subjects[num_train + num_test:]

def collect_files(subject_list):
    files = []
    for sid in subject_list:
        files.extend(subject_to_files[sid])
    return sorted(files)

train_files = collect_files(train_subjects)
test_files = collect_files(test_subjects)
val_files = collect_files(val_subjects)

train_path = output_dir / 'train_split_torus.json'
test_path = output_dir / 'test_split_torus.json'
val_path = output_dir / 'val_split_torus.json'

with train_path.open('w') as f:
    json.dump(train_files, f, indent=2)
with test_path.open('w') as f:
    json.dump(test_files, f, indent=2)
with val_path.open('w') as f:
    json.dump(val_files, f, indent=2)

print('Wrote:', train_path, test_path, val_path)
print(f'Train subjects/files: {len(train_subjects)}/{len(train_files)}')
print(f'Test subjects/files: {len(test_subjects)}/{len(test_files)}')
print(f'Val subjects/files: {len(val_subjects)}/{len(val_files)}')
print(f'Subject ratios: train {len(train_subjects)/num_subjects:.2%}, ' 
      f'test {len(test_subjects)/num_subjects:.2%}, ' 
      f'val {len(val_subjects)/num_subjects:.2%}')


Wrote: ../examples/splits/splits_torus/train_split_torus.json ../examples/splits/splits_torus/test_split_torus.json ../examples/splits/splits_torus/val_split_torus.json
Train subjects/files: 80/400
Test subjects/files: 15/75
Val subjects/files: 5/25
Subject ratios: train 80.00%, test 15.00%, val 5.00%


In [6]:
train_subjects_set = set(train_subjects)
test_subjects_set = set(test_subjects)
val_subjects_set = set(val_subjects)

assert train_subjects_set.isdisjoint(test_subjects_set)
assert train_subjects_set.isdisjoint(val_subjects_set)
assert test_subjects_set.isdisjoint(val_subjects_set)
assert train_subjects_set | test_subjects_set | val_subjects_set == set(subjects)

train_files_set = set(train_files)
test_files_set = set(test_files)
val_files_set = set(val_files)

assert train_files_set.isdisjoint(test_files_set)
assert train_files_set.isdisjoint(val_files_set)
assert test_files_set.isdisjoint(val_files_set)

all_split_files = train_files_set | test_files_set | val_files_set
all_subject_files = set()
for files in subject_to_files.values():
    all_subject_files.update(files)

assert all_split_files == all_subject_files

subject_to_split = {}
for sid in train_subjects:
    subject_to_split[sid] = 'train'
for sid in test_subjects:
    subject_to_split[sid] = 'test'
for sid in val_subjects:
    subject_to_split[sid] = 'val'

for sid, files in subject_to_files.items():
    split = subject_to_split[sid]
    if split == 'train':
        assert set(files).issubset(train_files_set)
    elif split == 'test':
        assert set(files).issubset(test_files_set)
    else:
        assert set(files).issubset(val_files_set)

print('Split checks passed: each subject is in exactly one split and all labeled files are covered.')


Split checks passed: each subject is in exactly one split and all labeled files are covered.
